# Raw activation normality analysis

Tests whether CLIP CLS-token activations at each layer follow a normal distribution,
analyzed separately for positive and negative examples of each concept.

**Two complementary perspectives on 1024-dim distributions:**
- **Marginal**: D'Agostino-Pearson test on each of the 1024 dimensions independently.
  Reports fraction of dimensions passing normality at α=0.05, plus mean skewness and
  excess kurtosis across dimensions.
- **PCA**: PCA to top-k principal components (k s.t. 95% variance explained),
  then same tests on the projections. Captures joint structure better than marginals.

Data used: **test split** (smaller, unbiased, sufficient n per concept group).

Output: `notebooks/results/distributions_analysis/raw_normality/`
- `results.csv` — one row per (concept, label, layer)
- heatmaps and layer curves

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import normaltest, skew, kurtosis
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Balanced concepts from download_data.ipynb
CONCEPTS = [
    'eyeglasses', 'bald', 'chubby', 'wearing_hat', 'male', 'smiling',
    'bags_under_eyes', 'blond_hair', 'double_chin', 'heavy_makeup',
]
NUM_LAYERS  = 24
ALPHA       = 0.05   # significance level for normality tests
PCA_VAR     = 0.95   # explained variance threshold for PCA
SPLIT       = 'test' # use test split for unbiased distribution estimates

RAW_DIR      = ROOT / 'data' / 'activations' / 'raw' / SPLIT
METADATA     = ROOT / 'data' / 'metadata.csv'
OUT_DIR      = ROOT / 'notebooks' / 'results' / 'distributions_analysis' / 'raw_normality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(),  f'Missing: {RAW_DIR}. Run 02_get_activations.ipynb first.'
assert METADATA.exists(), f'Missing: {METADATA}'

df_meta = pd.read_csv(METADATA)
df_split = df_meta[df_meta['split'] == SPLIT].reset_index(drop=True)
missing = [c for c in CONCEPTS if c not in df_split.columns]
assert not missing, f'Missing concept columns: {missing}'

print(f'Split    : {SPLIT} ({len(df_split)} images)')
print(f'Concepts : {CONCEPTS}')
print(f'Layers   : {NUM_LAYERS}')
print(f'Output   : {OUT_DIR}')

In [ ]:
def load_layer(layer_idx):
    """Load raw activations for SPLIT, merged with metadata. Returns DataFrame."""
    path = RAW_DIR / f'layer_{layer_idx:02d}.parquet'
    assert path.exists(), f'Missing: {path}'
    df = pd.read_parquet(path)
    return df.merge(df_split[['filename'] + CONCEPTS], on='filename', how='inner')


def get_feat_cols(df):
    return [c for c in df.columns if c not in ['filename'] + CONCEPTS]


def normality_stats(X):
    """Marginal normality stats over 1024-dim X (n_samples, 1024).
    Returns dict with scalar summary metrics.
    """
    n, d = X.shape
    if n < 8:  # normaltest requires at least 8 samples
        return None

    # ── Marginal (per dimension) ──
    _, pvals = normaltest(X, axis=0)              # (d,)
    frac_normal_marg = float((pvals > ALPHA).mean())
    mean_skew        = float(skew(X, axis=0).mean())
    mean_kurt        = float(kurtosis(X, axis=0).mean())  # excess kurtosis
    mean_abs_skew    = float(np.abs(skew(X, axis=0)).mean())
    mean_abs_kurt    = float(np.abs(kurtosis(X, axis=0)).mean())

    # ── PCA projection ──
    pca = PCA(n_components=min(n - 1, d))
    X_pca = pca.fit_transform(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    k = int(np.searchsorted(cumvar, PCA_VAR)) + 1
    k = min(k, X_pca.shape[1])
    X_k = X_pca[:, :k]

    _, pvals_pca = normaltest(X_k, axis=0)
    frac_normal_pca = float((pvals_pca > ALPHA).mean())
    mean_skew_pca   = float(skew(X_k, axis=0).mean())
    mean_kurt_pca   = float(kurtosis(X_k, axis=0).mean())

    return {
        'n':                n,
        'pca_k':            k,
        'pca_var_explained': float(cumvar[k - 1]),
        # marginal
        'frac_normal_marg': frac_normal_marg,
        'mean_skew':        mean_skew,
        'mean_kurt':        mean_kurt,
        'mean_abs_skew':    mean_abs_skew,
        'mean_abs_kurt':    mean_abs_kurt,
        # PCA
        'frac_normal_pca':  frac_normal_pca,
        'mean_skew_pca':    mean_skew_pca,
        'mean_kurt_pca':    mean_kurt_pca,
    }

In [ ]:
records = []

for layer_idx in tqdm(range(NUM_LAYERS), desc='Layers'):
    df = load_layer(layer_idx)
    feat_cols = get_feat_cols(df)
    X_all = df[feat_cols].values.astype(np.float32)

    for concept in CONCEPTS:
        labels = df[concept].values
        for label_val, label_name in [(1, 'pos'), (0, 'neg')]:
            mask = labels == label_val
            X_sub = X_all[mask]
            stats = normality_stats(X_sub)
            if stats is None:
                continue
            records.append({
                'layer': layer_idx,
                'concept': concept,
                'label': label_name,
                **stats,
            })

df_results = pd.DataFrame(records)
df_results.to_csv(OUT_DIR / 'results.csv', index=False)
print(f'Saved {len(df_results)} rows to results.csv')
df_results.head(4)

## Visualization

In [ ]:
# Heatmap: fraction of dimensions passing normality test (marginal)
# Averaged over pos and neg, per concept × layer

for metric, title in [
    ('frac_normal_marg', 'Fraction of dimensions normal (marginal, D\u2019Agostino-Pearson)'),
    ('frac_normal_pca',  'Fraction of PCs normal (PCA, D\u2019Agostino-Pearson)'),
    ('mean_abs_kurt',    'Mean |excess kurtosis| across dimensions'),
    ('mean_abs_skew',    'Mean |skewness| across dimensions'),
]:
    pivot = (
        df_results.groupby(['concept', 'layer'])[metric]
        .mean()
        .unstack('layer')
    )

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn' if 'frac' in metric else 'YlOrRd',
        annot=False, linewidths=0.3,
        vmin=0, vmax=1 if 'frac' in metric else None,
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Concept')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'heatmap_{metric}.png', dpi=150)
    plt.show()

In [ ]:
# Line plot: pos vs neg normality per concept, averaged over layers

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
axes = axes.flatten()

for ax, concept in zip(axes, CONCEPTS):
    sub = df_results[df_results['concept'] == concept]
    for label, ls in [('pos', '-'), ('neg', '--')]:
        s = sub[sub['label'] == label].sort_values('layer')
        ax.plot(s['layer'], s['frac_normal_marg'], ls=ls, label=label)
    ax.set_title(concept, fontsize=9)
    ax.set_xlabel('Layer', fontsize=8)
    ax.set_ylim(0, 1)
    ax.axhline(0.05, color='red', lw=0.8, ls=':', label='\u03b1=0.05')

axes[0].set_ylabel('Fraction normal (marginal)')
axes[0].legend(fontsize=8)
fig.suptitle('Fraction of dimensions passing normality per concept (pos vs neg)', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'curves_frac_normal_by_concept.png', dpi=150)
plt.show()

In [ ]:
# Summary: mean kurtosis and skewness across all concepts and labels

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, ylabel in [
    (axes[0], 'mean_abs_kurt', 'Mean |excess kurtosis|'),
    (axes[1], 'mean_abs_skew', 'Mean |skewness|'),
]:
    for label, ls in [('pos', '-'), ('neg', '--')]:
        s = (
            df_results[df_results['label'] == label]
            .groupby('layer')[metric].mean()
        )
        ax.plot(s.index, s.values, ls=ls, label=label)
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} averaged over concepts')
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'summary_kurt_skew.png', dpi=150)
plt.show()